# Frame-window metric comparison
Run all cells to score saved predictions, then compare methods across frame counts. Each point is an equal-weight mean over windows with status `ok`. K is the number of input frames. M counts skipped frames within a window. Start stride controls the separation between window starts.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "experiments/score_windows.py").is_file())
DATASET, SPLIT = "eds", "test"
EXPERIMENTS = {2: "windows_k2", 5: "windows_k5", 10: "windows_k10"}
M, START_STRIDE = 0, 50
METRICS = ["chamfer_full", "event_count_ratio", "iei_distance", "fano_factor", "voxel_mae"]
METHODS = []  # Empty discovers available methods for each experiment.
SEQUENCES = []  # Empty evaluates all sequences in the split.
WORKERS, DEVICE = 16, "cpu"
REPORTS = {}

## Compute metrics
Use the Rails kernel with the benchmark dependencies. Completed window scores are reused when the scorer's resume checks pass. The final line of each invocation identifies its report directory.

In [ ]:
REPORTS.clear()
for k, experiment in EXPERIMENTS.items():
    for metric in METRICS:
        command = [sys.executable, "-u", "experiments/score_windows.py", DATASET,
                   "--split", SPLIT, "--experiment-name", experiment, "--k", str(k),
                   "--m", str(M), "--start-stride", str(START_STRIDE),
                   "--metric", metric, "--workers", str(WORKERS), "--device", DEVICE]
        if METHODS:
            command.extend(["--methods", *METHODS])
        if SEQUENCES:
            command.extend(["--sequences", *SEQUENCES])
        print(f"Computing {metric} for K={k}", flush=True)
        last = ""
        with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, text=True) as proc:
            for line in proc.stdout:
                print(line, end="", flush=True)
                if line.strip():
                    last = line.strip()
            status = proc.wait()
        folder = Path(last)
        if not folder.is_absolute() or not all((folder / f).is_file() for f in ("summary.csv", "per_window.csv")):
            raise RuntimeError(f"Missing report for K={k}, {metric}. See output above.")
        REPORTS[k, metric] = folder
        if status:
            print("Scoring errors recorded. Inspect per-window status before interpreting means.")

## Means and coverage
Inspect valid and expected window counts alongside each mean. Different methods can have different valid windows even when their counts match.

In [ ]:
summary = pd.concat([pd.read_csv(p / "summary.csv") for p in REPORTS.values()], ignore_index=True)
windows = pd.concat([pd.read_csv(p / "per_window.csv") for p in REPORTS.values()], ignore_index=True)
display(summary[["k", "metric", "method", "mean", "unit", "n_valid", "n_total", "status_counts"]]
        .sort_values(["metric", "method", "k"]))

## Compare K within each metric
One line per method. Gaps indicate an unavailable mean. A dotted reference at 1 indicates the target for event-count ratio.

In [ ]:
ks = sorted(EXPERIMENTS)
methods = sorted(summary.method.unique())
colors = dict(zip(methods, plt.get_cmap("tab20").colors))
for metric in METRICS:
    rows = summary.loc[summary.metric == metric]
    fig, ax = plt.subplots(figsize=(9, 4))
    for method, group in rows.groupby("method"):
        values = group.set_index("k")["mean"].reindex(ks)
        if values.notna().any():
            ax.plot(ks, values, marker="o", color=colors[method], label=method)
    if metric == "event_count_ratio":
        ax.axhline(1, linestyle=":", color="grey")
    ax.set(xticks=ks, xlabel="Input frames per window (K)", ylabel=rows.unit.iloc[0], title=metric)
    if ax.get_legend_handles_labels()[0]:
        ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
    else:
        ax.text(.5, .5, "No valid scalar means", ha="center", transform=ax.transAxes)
    ax.grid(axis="y", alpha=.25)
    plt.tight_layout()
    plt.show()

## Inspect individual windows
Change the filters to inspect individual values and missing or partial scores.

In [ ]:
SHOW_K, SHOW_METRIC = min(EXPERIMENTS), METRICS[0]
SHOW_METHOD, SHOW_SEQUENCE = None, None
selected = windows[(windows.k == SHOW_K) & (windows.metric == SHOW_METRIC)]
if SHOW_METHOD:
    selected = selected[selected.method == SHOW_METHOD]
if SHOW_SEQUENCE:
    selected = selected[selected.sequence == SHOW_SEQUENCE]
display(selected.sort_values(["sequence", "first", "method"]))